# Module 4: Model Evaluation

In this notebook, we evaluate our models using the **Vertex AI Evaluation Service**. We will evaluate:
1. The Base model
2. The LoRA tuned model
3. The Full SFT tuned model

We will run evaluation jobs against our holdout evaluation datasets generated in Module 1, and log those metrics directly to our existing **Vertex AI Experiment** to objectively determine the best model.

In [ ]:
.pip install "google-cloud-aiplatform[evaluation]" python-dotenv -q

## 1. Setup and Authentication

In [ ]:
import os
from google.cloud import aiplatform
from vertexai.evaluation import EvalTask
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

PROJECT_ID = os.getenv("PROJECT_ID", "YOUR_PROJECT_ID")
LOCATION = os.getenv("LOCATION", "us-central1")
STAGING_BUCKET = os.getenv("STAGING_BUCKET", "YOUR_GCS_BUCKET_NAME")

if not STAGING_BUCKET.startswith("gs://"):
    STAGING_BUCKET = f"gs://{STAGING_BUCKET}"

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Staging Bucket: {STAGING_BUCKET}")

## 2. Re-attach to Vertex Experiment

In [ ]:
EXPERIMENT_NAME = os.getenv("EXPERIMENT_NAME", "cyber-tuning-demo")
TENSORBOARD_NAME = "cyber-tuning-tb"

# Retrieve existing TensorBoard
tb_instances = aiplatform.Tensorboard.list(filter=f'display_name="{TENSORBOARD_NAME}"')
if tb_instances:
    tb_instance = tb_instances[0]
    aiplatform.init(experiment=EXPERIMENT_NAME, experiment_tensorboard=tb_instance)
    print(f"Attached to Experiment: {EXPERIMENT_NAME}")
else:
    print("ERROR: TensorBoard instance not found. Run Module 2 first.")

## 3. Run Evaluation against Test Dataset

We will evaluate using Vertex AI's managed rapid evaluation system. We provide the dataset and the exact metrics to compute (e.g., `rouge_l` for text similarity against ground truth, or `coherence` / `instruction_following` for LLM-as-a-judge).

In [ ]:
import pandas as pd
import json

BASE_MODEL_ID = "meta/llama3_1@llama-3.1-8b"
EVAL_DATA_URI = "gs://YOUR_BUCKET_NAME/tuning_data/your_file.jsonl"

print("Loading evaluation dataset..")
df = pd.read_json(EVAL_DATA_URI, lines=True)

eval_data = []
for idx, row in df.iterrows():
    messages = row['messages']
    prompt = messages[0]['content'] + "\n\n" + messages[1]['content']
    reference = messages[2]['content']
    eval_data.append({"prompt": prompt, "reference": reference})

eval_df = pd.DataFrame(eval_data)

eval_task = EvalTask(
    dataset=eval_df,
    metrics=["rouge_l_sum", "exact_match", "bleu"], 
    experiment=EXPERIMENT_NAME
)

def evaluate_model(model_name: str, run_name: str):
    print(f"\nEvaluating: {run_name}..")
    try:
        result = eval_task.evaluate(
            model=model_name,
            experiment_run_name=run_name
        )
        print(f"Results for {run_name}:")
        for metric, value in result.summary_metrics.items():
            print(f"- {metric}: {value}")
    except Exception as e:
        print(f"Evaluation error for {run_name}: {e}")

evaluate_model(BASE_MODEL_ID, "eval-baseline")


## 4. Compare Evaluation Metrics

Return to **Vertex AI -> Experiments** in the Google Cloud Console. Select the 3 runs: `eval-baseline`, `run-lora-1`, and `run-full-sft-1`. 

Click **Compare**. You will see a detailed dashboard comparing the ROUGE and exact match scores side-by-side to justify which tuning method produced the superior cybersecurity model.